In [87]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import vitaldb

In [88]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [89]:
from src.preprocessing import process_case, TRACK_NAMES

print("Multi-patient pipeline ready!")

Multi-patient pipeline ready!


In [90]:
caseids = vitaldb.find_cases(TRACK_NAMES)

print("Number of matching cases:", len(caseids))
print("First 10 case IDs:", caseids[:10])

C:\Users\samat\AppData\Local\Temp\ipykernel_2532\1608594304.py:1: DeprecationWarning: find_cases() relies on the per-track 'trks' index, which is deprecated and may be removed in a future release.
  caseids = vitaldb.find_cases(TRACK_NAMES)


Number of matching cases: 3730
First 10 case IDs: [1, 3, 4, 7, 10, 12, 13, 14, 16, 17]


In [91]:
test_caseids = caseids[:5]

print("Cases selected for pipeline testing:")
print(test_caseids)

Cases selected for pipeline testing:
[1, 3, 4, 7, 10]


In [92]:
processed_cases = []
failed_cases = []

for caseid in test_caseids:

    print(f"Processing case {caseid}...")

    try:
        df_case = process_case(caseid)

        processed_cases.append(df_case)

        print(
            f"Case {caseid} completed - "
            f"{len(df_case)} rows"
        )

    except Exception as e:

        failed_cases.append({
            "caseid": caseid,
            "error": str(e)
        })

        print(
            f"Case {caseid} failed: {e}"
        )

Processing case 1...
Case 1 completed - 193 rows
Processing case 3...
Case 3 completed - 74 rows
Processing case 4...
Case 4 completed - 350 rows
Processing case 7...
Case 7 completed - 263 rows
Processing case 10...
Case 10 completed - 350 rows


In [93]:
print("Successful cases:", len(processed_cases))
print("Failed cases:", len(failed_cases))

Successful cases: 5
Failed cases: 0


In [94]:
failed_cases

[]

In [95]:
df_multi = pd.concat(
    processed_cases,
    ignore_index=True
)

In [96]:
print("Combined shape:", df_multi.shape)

Combined shape: (1230, 23)


In [97]:
df_multi.head()

,minute,heart_rate,spo2,respiratory_rate,mean_arterial_pressure,spo2_valid_count,samples_in_minute,spo2_coverage,future_spo2_5min,spo2_lag_1,...,spo2_change_3min,spo2_rolling_mean_5,spo2_rolling_std_5,heart_rate_lag_1,heart_rate_rolling_mean_5,rr_lag_1,rr_rolling_mean_5,map_lag_1,map_rolling_mean_5,caseid
0,0,79.0,97.0,NaN,NaN,29,30,0.966667,100.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,1,85.0,97.0,NaN,NaN,30,30,1.000000,100.0,97.0,...,NaN,NaN,NaN,79.0,NaN,NaN,NaN,NaN,NaN,1
2,2,88.0,97.0,NaN,NaN,30,30,1.000000,100.0,97.0,...,NaN,97.00,0.00000,85.0,84.00,NaN,NaN,NaN,NaN,1
3,3,87.0,98.0,NaN,NaN,30,30,1.000000,100.0,97.0,...,1.0,97.25,0.50000,88.0,84.75,NaN,NaN,NaN,NaN,1
4,4,86.0,100.0,NaN,NaN,30,30,1.000000,100.0,98.0,...,3.0,97.80,1.30384,87.0,85.00,NaN,NaN,NaN,NaN,1


In [98]:
df_multi["caseid"].value_counts()

caseid
4     350
10    350
7     263
1     193
3      74
Name: count, dtype: int64

In [99]:
print(
    "Unique patients:",
    df_multi["caseid"].nunique()
)

Unique patients: 5


In [100]:
print(
    "Rows with target:",
    df_multi["future_spo2_5min"].notna().sum()
)

print(
    "Rows without target:",
    df_multi["future_spo2_5min"].isna().sum()
)

Rows with target: 1162
Rows without target: 68


In [101]:
df_multi["future_spo2_5min"].describe()

count    1162.00000
mean       99.39716
std         1.62054
min        77.00000
25%        99.00000
50%       100.00000
75%       100.00000
max       100.00000
Name: future_spo2_5min, dtype: float64

In [102]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

In [103]:
spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

In [104]:
all_vital_features = spo2_hr_features + [
    "respiratory_rate",
    "rr_lag_1",
    "rr_rolling_mean_5",
    "mean_arterial_pressure",
    "map_lag_1",
    "map_rolling_mean_5"
]

In [105]:
def count_complete_rows(df, features):

    required_columns = (
        features +
        ["future_spo2_5min"]
    )

    return (
        df[required_columns]
        .dropna()
        .shape[0]
    )

In [106]:
print(
    "SpO2-only usable rows:",
    count_complete_rows(
        df_multi,
        spo2_features
    )
)

print(
    "SpO2 + HR usable rows:",
    count_complete_rows(
        df_multi,
        spo2_hr_features
    )
)

print(
    "All vitals usable rows:",
    count_complete_rows(
        df_multi,
        all_vital_features
    )
)

SpO2-only usable rows: 1132
SpO2 + HR usable rows: 1125
All vitals usable rows: 1023


In [107]:
output_file = (
    project_root /
    "data" /
    "processed" /
    "multi_patient_test.csv"
)

df_multi.to_csv(
    output_file,
    index=False
)

print("Saved to:")
print(output_file)

Saved to:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\data\processed\multi_patient_test.csv


In [ ]:
failed_file = (
    project_root /
    "data" /
    "processed" /
    "failed_cases_50.csv"
)

pd.DataFrame(
    failed_cases
).to_csv(
    failed_file,
    index=False
)

print("Failure log saved:")
print(failed_file)

Failure log saved:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\data\processed\failed_cases_50.csv


: 